# Struktur zwischen den Quadrupeln und E8 – Sage Notebook

Dieses Notebook setzt die Inhalte aus **Struktur_zwischen_zwei_Quadrupeln.md** und dem **Schütte/E8-Programm** in SageMath um:
- Zwei begrenzende Quadrupel (N_low, N_high) für eine Zielzahl M
- Drei Trajektorien (linear, diskret, logarithmisch) und Geodäten (Alice, Father, Taurus)
- Father-Geodäte: 16 Quadrupel-Produkte, GCD mit n
- RSA-Demo: schwache Testschlüssel, E8-artige Faktorisierung, Entschlüsselung

## 1. Imports und Familien E, A, B, C (mod 12)

In [ ]:
from sage.all import Integer, next_prime, is_prime, gcd, factor, inverse_mod, power_mod, prod, randint

# Familien: Rest mod 12 (E, A, B, C)
RESIDUES = (1, 5, 7, 11)  # E, A, B, C
FAM_NAMES = {1: 'E', 5: 'A', 7: 'B', 11: 'C'}

def fam(p):
    r = int(p % 12)
    return FAM_NAMES.get(r, '?')

def next_prime_in_class(start, residue, direction=+1):
    '''Nächste Primzahl p mit p ≡ residue (mod 12). direction +1 = aufwärts.'''
    p = Integer(start)
    if direction > 0:
        r = p % 12
        diff = (residue - r) % 12
        if diff == 0 and p > 0:
            p += 12
        else:
            p += diff
        while p >= 2:
            if is_prime(p):
                return p
            p += 12
    else:
        r = p % 12
        diff = (r - residue) % 12
        if diff == 0:
            p -= 12
        else:
            p -= diff
        while p >= 2:
            if is_prime(p):
                return p
            p -= 12
    return None

print('Familien E, A, B, C: Reste mod 12 =', RESIDUES)

: 

## 2. Zwei begrenzende Quadrupel N_low, N_high für M

In [ ]:
def largest_prime_in_class(bound, residue):
    '''Größte Primzahl p ≡ residue (mod 12) mit p ≤ bound.'''
    p = (bound // 12) * 12 + residue
    if p > bound:
        p -= 12
    while p >= 2:
        if is_prime(p):
            return Integer(p)
        p -= 12
    return None

def smallest_prime_in_class(bound, residue):
    '''Kleinste Primzahl p ≡ residue (mod 12) mit p ≥ bound.'''
    r = bound % 12
    diff = (residue - r) % 12
    p = bound + diff
    if p < 2:
        p = max(2, 13 if residue == 1 else residue)
    while True:
        if is_prime(p):
            return Integer(p)
        p += 12

def lower_quadrupel(M):
    '''Größtes Quadrupel N_low ≤ M (eine Primzahl pro Familie E,A,B,C).'''
    M = Integer(M)
    t = int(M ** 0.25)
    if t < 12:
        t = 12
    primes = [largest_prime_in_class(t, r) for r in RESIDUES]
    if None in primes:
        return None, None
    N = prod(primes)
    while N > M and t > 2:
        t -= 1
        primes = [largest_prime_in_class(t, r) for r in RESIDUES]
        if None in primes:
            return None, None
        N = prod(primes)
    return N, tuple(primes)

def upper_quadrupel(M):
    '''Kleinstes Quadrupel N_high ≥ M.'''
    M = Integer(M)
    t = int(M ** 0.25) + 1
    primes = [smallest_prime_in_class(t, r) for r in RESIDUES]
    N = prod(primes)
    while N < M:
        t += 1
        primes = [smallest_prime_in_class(t, r) for r in RESIDUES]
        N = prod(primes)
    return N, tuple(primes)

# Demo: M = 2^80 - 1
M_demo = 2**80 - 1
N_low, quad_low = lower_quadrupel(M_demo)
N_high, quad_high = upper_quadrupel(M_demo)
print('M = 2^80 - 1 =', M_demo)
print('N_low =', quad_low, '=> Produkt =', N_low)
print('N_high =', quad_high, '=> Produkt =', N_high)
print('N_low ≤ M ≤ N_high:', N_low <= M_demo <= N_high)

## 3. Drei Trajektorien (Alice/Morley, Father/Walter, Taurus/Kepler)

In [ ]:
# Trajektorie 1: Linear (Alice/Morley)
def T1(lam, N_low, N_high):
    return N_low + lam * (N_high - N_low)

# Trajektorie 3: Logarithmisch (Taurus/Kepler)
def T3(mu, N_low, N_high):
    return (N_low ** (1 - mu)) * (N_high ** mu)

# Beispiel: lambda = 0.5, mu = 0.5
lam = 0.5
mu = 0.5
print('T1(0.5) =', T1(lam, N_low, N_high))
print('T3(0.5) =', T3(mu, N_low, N_high))
print('M liegt zwischen N_low und N_high.')

## 4. Father-Geodäte: Q_low, Q_high und 16 Quadrupel-GCD (E8-artig)

In [ ]:
def get_Q_low_Q_high(n):
    '''Für n: nächste Primzahlen in jeder Familie unterhalb (Q_low) und oberhalb (Q_high) von n^(1/4).'''
    n = Integer(n)
    t = int(n ** 0.25)
    if t < 12:
        t = 12
    Q_low = [largest_prime_in_class(t, r) for r in RESIDUES]
    Q_high = [smallest_prime_in_class(t + 1, r) for r in RESIDUES]
    if None in Q_low or None in Q_high:
        return None, None
    return Q_low, Q_high

def father_geodesic_gcd(n, Q_low, Q_high):
    '''16 Quadrupel-Produkte aus Q_low/Q_high, GCD mit n. Gibt Faktor zurück falls 1 < g < n.'''
    n = Integer(n)
    for mask in range(16):
        P = 1
        for i in range(4):
            P *= Q_high[i] if (mask >> i) & 1 else Q_low[i]
        g = gcd(P, n)
        if g > 1 and g < n:
            return g
    return None

# Demo: n = kleines Semiprim (schwacher RSA-Modul)
small_p = 131
q = next_prime(10**20)  # großer Primfaktor
n_demo = small_p * q
Q_low, Q_high = get_Q_low_Q_high(n_demo)
if Q_low and Q_high:
    fac = father_geodesic_gcd(n_demo, Q_low, Q_high)
    print('n =', small_p, '* q (Semiprim, n hat', n_demo.nbits(), 'Bits)')
    print('Father-Geodäte (16 Produkte): Faktor gefunden =', fac)
    if fac:
        print('GCD(n, P) =', fac, ', anderer Faktor =', n_demo // fac)
else:
    print('Q_low/Q_high nicht berechenbar für dieses n.')

## 5. RSA-Demo: schwache Testschlüssel, Faktorisierung (Trial + Father), Entschlüsselung

In [ ]:
def trial_divison_factor(n, limit=2000):
    '''Kleine Faktoren durch Trial Division (E8 erster Schritt).'''
    n = Integer(n)
    for f in range(2, min(limit, n.isqrt() + 2)):
        if f % 2 == 0 and f > 2:
            continue
        if n % f == 0:
            return f
    return None

def e8_style_factor(n):
    '''E8-artige Faktorisierung: Trial Division, dann Father (16 Quadrupel-GCD).'''
    n = Integer(n)
    f = trial_divison_factor(n)
    if f:
        return f
    Q_low, Q_high = get_Q_low_Q_high(n)
    if Q_low and Q_high:
        f = father_geodesic_gcd(n, Q_low, Q_high)
        if f:
            return f
    return None

def rsa_demo(num_keys=50):
    '''RSA-Demo: num_keys schwache RSA-Moduln (n = small_p * q), E8-artig faktorisieren, entschlüsseln.'''
    small_primes = [101, 103, 107, 109, 113, 127, 131, 137, 139, 149, 151, 157, 163, 167, 173]
    e = 65537
    factored = 0
    decrypted_ok = 0
    for i in range(num_keys):
        small_p = small_primes[i % len(small_primes)]
        q = next_prime(2^200 + i * 2^50)  # 200-Bit-Bereich, variiert
        n = small_p * q
        m = randint(2, n - 2)  # Klartext
        c = power_mod(m, e, n)  # Chiffrat
        f = e8_style_factor(n)
        if f:
            factored += 1
            p = f
            q_other = n // p
            phi = (p - 1) * (q_other - 1)
            if gcd(e, phi) == 1:
                d = inverse_mod(Integer(e), phi)
                m_dec = power_mod(c, d, n)
                if m_dec == m:
                    decrypted_ok += 1
    return factored, decrypted_ok

print('RSA-Demo: 50 schwache Testschlüssel (n = kleiner_Faktor * q), E8-artige Faktorisierung, Entschlüsselung.')
factored, dec_ok = rsa_demo(50)
print('Ergebnis: 50 Schlüssel erzeugt,', factored, 'faktorisiert,', dec_ok, 'erfolgreich entschlüsselt.')

## 6. Kurzfassung

- **Struktur zwischen den Quadrupeln:** N_low ≤ M ≤ N_high (zwei begrenzende Quadrupel).
- **Drei Trajektorien:** Linear (Alice), diskret im Gitter (Father), logarithmisch (Taurus).
- **Father-Geodäte:** 16 Produkte aus Q_low/Q_high, GCD mit n (E8-artig).
- **RSA-Demo:** Schwache Testschlüssel werden lokal erzeugt, E8-artig faktorisiert und entschlüsselt (keine externen Schlüssel).